### 1. Load DataSet

In [9]:
import dagshub
dagshub.init(repo_owner='iamdebasishdas123', repo_name='YouTube-Mood-Tracker', mlflow=True)
import mlflow

Accessing as iamdebasishdas123

Initialized MLflow to track repo "iamdebasishdas123/YouTube-Mood-Tracker"

Repository iamdebasishdas123/YouTube-Mood-Tracker initialized!

c:\Users\Debasish Das\Desktop\sentiment_analysis\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
import numpy as np
import pandas as pd
# Load the data
def load_data():
    df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
    return df
df = load_data()
df.head()

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


### 2. Preprocessing of Data

In [11]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


nltk.download('stopwords')
nltk.download('wordnet')

# Define the preprocessing function
def preprocess_comment(comment):
    # Convert to lowercase
    comment = comment.lower()

    # Remove trailing and leading whitespaces
    comment = comment.strip()

    # Remove newline characters
    comment = re.sub(r'\n', ' ', comment)

    # Remove non-alphanumeric characters, except punctuation
    comment = re.sub(r'[^A-Za-z0-9\s!?.,]', '', comment)

    # Remove stopwords but retain important ones for sentiment analysis
    stop_words = set(stopwords.words('english')) - {'not', 'but', 'however', 'no', 'yet'}
    comment = ' '.join([word for word in comment.split() if word not in stop_words])

    # Lemmatize the words
    lemmatizer = WordNetLemmatizer()
    comment = ' '.join([lemmatizer.lemmatize(word) for word in comment.split()])

    return comment

[nltk_data] Downloading package stopwords to C:\Users\Debasish
[nltk_data]     Das\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Debasish
[nltk_data]     Das\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [12]:
def preprocess_data(df):
    df=df.dropna()
    df=df.drop_duplicates()
    df = df[~(df['clean_comment'].str.strip() == '')]
    df['clean_comment'] = df['clean_comment'].apply(preprocess_comment)
    return df

In [13]:
data=preprocess_data(df)

#### Experiments Embedding Model

In [11]:
# Set or create an experiment
mlflow.set_experiment("Embedding_Experiments")

<Experiment: artifact_location='mlflow-artifacts:/f8195b3b2d90485782486f3af3a211cc', creation_time=1783534617348, experiment_id='1', last_update_time=1783534617348, lifecycle_stage='active', name='Embedding_Experiments', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [14]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec
import numpy as np

# Define experiment parameters
ngram_ranges = [ (1,2), (1,3)]  # unigram, bigram, trigram
vectorizers = {
    'bow': CountVectorizer(),
    'tfidf': TfidfVectorizer(),
    # 'word2vec': None  # Will be handled separately
}
feature_sizes = [5000, 7000, 9000, 11000, 13000]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    data['clean_comment'], data['category'], test_size=0.2, random_state=42
)
y_train = y_train.map({-1: 0, 0: 1, 1: 2})
y_test = y_test.map({-1: 0, 0: 1, 1: 2})

# Run experiments
for ngram_range in ngram_ranges:
    for vectorizer_name, vectorizer in vectorizers.items():
        for feature_size in feature_sizes:
            run_name = f"{vectorizer_name}_ngram{ngram_range}_features{feature_size}"
            with mlflow.start_run(run_name=run_name):
                # Set experiment parameters
                mlflow.log_params({
                    'ngram_range': ngram_range,
                    'vectorizer': vectorizer_name,
                    'feature_size': feature_size
                })

                # Handle different vectorizers
                if vectorizer_name in ['bow', 'tfidf']:
                    # Configure vectorizer
                    vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=feature_size) \
                        if vectorizer_name == 'bow' \
                        else TfidfVectorizer(ngram_range=ngram_range, max_features=feature_size)
                    
                    # Transform data
                    X_train_vec = vectorizer.fit_transform(X_train)
                    X_test_vec = vectorizer.transform(X_test)
                
                # Word2Vec implementation
                elif vectorizer_name == 'word2vec':
                    # Train Word2Vec model
                    tokenized_comments = [comment.split() for comment in X_train]
                    w2v_model = Word2Vec(tokenized_comments, vector_size=100, window=5, min_count=5, workers=4)
                    
                    # Create sentence vectors
                    def average_word_vectors(texts, model):
                        return np.array([
                            np.mean([model.wv[word] for word in text.split() if word in model.wv], axis=0)
                            for text in texts
                        ])
                    
                    X_train_vec = average_word_vectors(X_train, w2v_model)
                    X_test_vec = average_word_vectors(X_test, w2v_model)
                    
                # Train XGBoost model
                model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
                model.fit(X_train_vec, y_train)
                
                # Evaluate model with accuracy, precision, recall, and F1 score
                y_pred = model.predict(X_test_vec)
                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred, average='weighted')
                recall = recall_score(y_test, y_pred, average='weighted')
                f1 = f1_score(y_test, y_pred, average='weighted')
                
                
                # Log metrics
                mlflow.log_metrics({
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1_score': f1
                })
                
                print(f"Completed experiment: {vectorizer_name}, ngram={ngram_range}, features={feature_size}, Accuracy={accuracy:.4f}")

Completed experiment: bow, ngram=(1, 2), features=5000, Accuracy=0.7466
🏃 View run bow_ngram(1, 2)_features5000 at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/1/runs/e108695b8ee74c908cc6c0f1dccd69db
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/1
Completed experiment: bow, ngram=(1, 2), features=7000, Accuracy=0.7475
🏃 View run bow_ngram(1, 2)_features7000 at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/1/runs/5435923ba4da477b903a2a192ccca8d3
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/1
Completed experiment: bow, ngram=(1, 2), features=9000, Accuracy=0.7494
🏃 View run bow_ngram(1, 2)_features9000 at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/1/runs/35de440a4a9a41269193b1c199214624
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlf

#### After Experiment Finding
- Vectorizer TFIDF
- Feture Size: 13000 
- ngram Size: (1,1)

### Experiments ML Algorithm

- 1. LightBGM 
- 2. SVM 
- 3. Logistic Regression
- 4. Xgboost

In [1]:
!pip install lightgbm



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from lightgbm import LGBMClassifier
def light_gbm_model():
    return LGBMClassifier(n_estimators=100,max_depth=10, learning_rate=0.05, random_state=42)

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

def logistic_model(strategy='multinomial'):
    """
    Initializes a Logistic Regression model for multi-class classification.
    
    Parameters:
    strategy (str): 'multinomial' for native global softmax behavior,
                    or 'ovr' to force a One-vs-Rest wrapper strategy.
    """
    # Base model handles 'multinomial' natively when multi-class data is supplied
    base_model = LogisticRegression(solver='lbfgs')
    
    if strategy == 'ovr':
        return OneVsRestClassifier(base_model)
        
    return base_model




In [6]:
from sklearn.svm import SVC
def svm_model(kernel='rbf', C=1.0):
    model=SVC(kernel=kernel, C=C, gamma='scale')
    return model

In [7]:
from sklearn.ensemble import RandomForestClassifier
def random_forest_model(n_estimators=100, max_depth=10):
    model=RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    return model

In [8]:
from xgboost import XGBClassifier
def xgboost_model(n_estimators=350, max_depth=12, learning_rate=0.1):
    model=XGBClassifier(n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss')
    return model

In [18]:
# Run grid of experiments: BOW and TF-IDF with multiple feature sizes and models, all logged to MLflow
import os
import time
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
# Set or create an experiment
mlflow.set_experiment("Model_Experiments")
# Feature sizes to evaluate
feature_size=13000
ngram_ranges = (1, 1)
encoders = {
    'tfidf': TfidfVectorizer
}

# Model constructors defined earlier in the notebook (reuse     those functions)
model_constructors = {
    # "LightGBM": lambda: light_gbm_model(),
    "LogisticRegression": lambda: logistic_model(),
    "SVM": lambda: svm_model(),
    "RandomForest": lambda: random_forest_model(),
    "XGBClassifier": lambda: xgboost_model()
}   


results = []
corpus = df['clean_comment'].fillna('').astype(str)
# ensure labels are mapped consistently to 0/1/2
labels = df['category'].map({-1: 0, 0: 1, 1: 2})

# Run experiments
for vec_name, VecClass in encoders.items():
    for n_features in [feature_size]:
        print(f'=== Vectorizer: {vec_name}, features: {n_features} ===')
        vectorizer = VecClass(max_features=n_features, ngram_range=ngram_ranges)
        X = vectorizer.fit_transform(corpus)
        y = labels

        for model_name, make_model in model_constructors.items():
            run_name = f'{model_name}_{vec_name}_{n_features}'
            print('Run:', run_name)
            with mlflow.start_run(run_name=run_name):
                # Tags and params
                mlflow.set_tag('experiment_type', 'grid_search')
                mlflow.set_tag('model_family', model_name)
                mlflow.log_param('vectorizer', vec_name)
                mlflow.log_param('vectorizer_max_features', n_features)
                mlflow.log_param('run_started_at', time.strftime('%Y-%m-%d %H:%M:%S'))

                model = make_model()
                # Try to log model hyperparameters when possible
                try:
                    mlflow.log_params(model.get_params())
                except Exception:
                    pass

                # Split and train
                X=X.astype(float)  # Ensure X is in float format for compatibility with models
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

                # Metrics
                acc = accuracy_score(y_test, y_pred)
                mlflow.log_metric('accuracy', float(acc))

                crep = classification_report(y_test, y_pred, output_dict=True)
                for label, metrics in crep.items():
                    if isinstance(metrics, dict):
                        clean_label = str(label).replace(' ', '_')
                        for metric, value in metrics.items():
                            try:
                                mlflow.log_metric(f'{clean_label}_{metric}', float(value))
                            except Exception:
                                pass

                # Confusion matrix artifact
                cm = confusion_matrix(y_test, y_pred)
                plt.figure(figsize=(6, 5))
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
                plt.xlabel('Predicted')
                plt.ylabel('Actual')
                plt.title(f'Confusion Matrix: {run_name}')
                fname = f'confusion_{run_name}.png'
                plt.savefig(fname)
                plt.close()
                mlflow.log_artifact(fname)

                # Log model artifact (use native logger for xgboost)
                try:
                    if model_name == 'XGBoost':
                        mlflow.xgboost.log_model(model, 'model')
                    else:
                        mlflow.sklearn.log_model(model, 'model')
                except Exception:
                    pass

                # Save a small dataset snapshot for reproducibility
                try:
                    sample_name = f'dataset_snapshot_{run_name}.csv'
                    pd.concat([pd.Series(y_test, name='y_test').reset_index(drop=True), pd.DataFrame(X_test.toarray())], axis=1).to_csv(sample_name, index=False)
                    mlflow.log_artifact(sample_name)
                except Exception:
                    # skip if X_test is large or not convertible to array
                    pass

                # Record summary for later comparison
                results.append({'run': run_name, 'vectorizer': vec_name, 'n_features': n_features, 'model': model_name, 'accuracy': float(acc)})

# Summarize results across all runs
res_df = pd.DataFrame(results)
if not res_df.empty:
    res_df = res_df.sort_values(['model', 'vectorizer', 'n_features'], ascending=[True, True, True]).reset_index(drop=True)
    display(res_df)
else:
    print('No results recorded.')

=== Vectorizer: tfidf, features: 13000 ===
Run: LogisticRegression_tfidf_13000


2026/07/28 23:05:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/28 23:05:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LogisticRegression_tfidf_13000 at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/2/runs/62b2ee3800bc49e6beacd0a37a3543c8
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/2
Run: SVM_tfidf_13000


2026/07/28 23:26:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/28 23:27:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run SVM_tfidf_13000 at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/2/runs/63459cbec3514123bd7ce25db174f9f6
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/2


KeyboardInterrupt: 

### Tuning Hyperparameter